# 07 - Scaling Reproduction: RobustScaler Blow-Up vs. StandardScaler Clipping

## Problem Statement & Research Context
During earlier stages of model experimentation, deep learning models (1D-CNN) exhibited catastrophic performance degradation (~21% to ~60% accuracy, near-zero Macro F1) when trained on network flow features scaled with `RobustScaler`. 

### The Root Cause:
1. **Severe Right-Skew & Zero IQR**: Network traffic metrics like `Active Std`, `Idle Std`, and flag counts have massive zero-inflation (median = 0, $Q_1 = 0$, $Q_3 = 0$), meaning their interquartile range ($IQR = Q_3 - Q_1$) is exactly 0.
2. **RobustScaler Behavior**: In Scikit-Learn, when $IQR = 0$, `RobustScaler` substitutes a scaling factor of 1.0 to avoid division by zero:
   $$\hat{x} = \frac{x - \text{median}}{1.0} = x$$
   Consequently, raw outlier values up to **74,200,000** pass directly into the neural network unscaled!
3. **Near-Zero IQRs**: For features where IQR is small but non-zero (e.g., $IQR = 48$ on raw max of $1.2 \times 10^8$), division by IQR produces scaled magnitudes exceeding $2.5 \times 10^6$.
4. **Gradient Collapse**: When unclipped inputs on the order of $10^6 - 10^7$ pass into 1D-convolutions and batch normalization, activations explode, batch variances destabilize, and the network collapses into predicting only the majority class (BENIGN).

### The Solution:
Switching to `StandardScaler` combined with explicit outlier clamping:
$$\hat{x}_{\text{clipped}} = \text{clip}\left(\frac{x - \mu}{\sigma}, -10, 10\right)$$
ensures that standard deviations are strictly $\approx 0.85$ and no activation ever exceeds $[-10, 10]$, allowing the 1D-CNN to converge smoothly to **>96% accuracy** and **>0.90 Macro F1**.

This notebook provides the rigorous, isolated reproduction experiment documenting both configurations on the exact same 70/15/15 stratified split (Seed 42).


In [1]:
import os
import sys
import time
import copy
import json
import platform
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

try:
    from IPython.display import display
except ImportError:
    display = print

import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__} | Device: {device}")
print(f"Platform: {platform.system()} {platform.release()} | Processor: {platform.processor()}")


PyTorch Version: 2.14.0+cpu | Device: cpu
Platform: Windows 11 | Processor: AMD64 Family 25 Model 68 Stepping 1, AuthenticAMD


## 1. Load Dataset & Stratified 70/15/15 Split (Seed 42)
We load `Datasets/CICIDS_clean_subset.parquet` (169,686 rows, 44 network flow features, 7 classes) and replicate the exact 70/15/15 stratified split from Notebook 05 with random seed 42.


In [2]:
t0 = time.time()
df = pd.read_parquet('../Datasets/CICIDS_clean_subset.parquet')
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns in {time.time()-t0:.2f}s")

X_all = df.drop(columns=['Label'])
y_all_raw = df['Label']

label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(y_all_raw)
classes = list(label_encoder.classes_)
num_classes = len(classes)

# Exact 70% Train, 15% Validation, 15% Test Stratified Split (Seed 42)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, stratify=y_all, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Split sizes -> Train: {len(X_train):,} (70%) | Val: {len(X_val):,} (15%) | Test: {len(X_test):,} (15%)")
print(f"Target classes ({num_classes}): {classes}")


Loaded 169,686 rows and 45 columns in 0.06s


Split sizes -> Train: 118,780 (70%) | Val: 25,453 (15%) | Test: 25,453 (15%)
Target classes (7): ['BENIGN', 'Bot', 'Brute Force', 'DDoS', 'DoS', 'PortScan', 'Web Attack']


## 2. RobustScaler: Feature Distribution Pathology & IQR Analysis
We fit `RobustScaler` strictly on `X_train` and transform both train and test without clipping.
We then compute:
1. The **min, max, and overall std** of the scaled train and test data.
2. The **10 features with the smallest IQR**.
3. The **top features with extreme scaled values** to identify where the blow-up originates.


In [3]:
# 1. Fit RobustScaler on train only
rs = RobustScaler()
X_train_rs = rs.fit_transform(X_train)
X_val_rs = rs.transform(X_val)
X_test_rs = rs.transform(X_test)

# Convert to DataFrame for feature-level diagnostics
X_train_rs_df = pd.DataFrame(X_train_rs, columns=X_train.columns)
X_test_rs_df = pd.DataFrame(X_test_rs, columns=X_test.columns)

print("="*85)
print("ROBUSTSCALER (NO CLIPPING) - SUMMARY STATISTICS")
print("="*85)
print(f"Train Scaled Min: {X_train_rs.min():15.4e} | Max: {X_train_rs.max():15.4e} | Std: {X_train_rs.std():15.4e}")
print(f"Test  Scaled Min: {X_test_rs.min():15.4e} | Max: {X_test_rs.max():15.4e} | Std: {X_test_rs.std():15.4e}")

# 2. 10 Features with the smallest IQR
# In scikit-learn RobustScaler: rs.scale_ stores the IQR (or 1.0 if raw IQR == 0)
# We also compute the exact raw IQR (Q75 - Q25) for complete academic transparency
raw_iqr = X_train.quantile(0.75) - X_train.quantile(0.25)
smallest_iqr_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Raw_IQR (Q75-Q25)': raw_iqr.values,
    'RobustScaler_scale_': rs.scale_,
    'Median': rs.center_,
    'Raw_Min': X_train.min().values,
    'Raw_Max': X_train.max().values,
    'Scaled_Max': X_train_rs_df.max().values
}).sort_values(by=['Raw_IQR (Q75-Q25)', 'Scaled_Max'], ascending=[True, False]).reset_index(drop=True)

print("\n" + "="*85)
print("10 FEATURES WITH THE SMALLEST IQR (AND THEIR SCALED IMPACT)")
print("="*85)
smallest_10_iqr = smallest_iqr_df.head(10)
print(smallest_10_iqr[['Feature', 'Raw_IQR (Q75-Q25)', 'RobustScaler_scale_', 'Median', 'Raw_Max', 'Scaled_Max']].to_string(index=False))

print("\n" + "="*85)
print("TOP 10 FEATURES WITH LARGEST SCALED MAXIMUMS IN ROBUSTSCALER")
print("="*85)
top_scaled_max = smallest_iqr_df.sort_values(by='Scaled_Max', ascending=False).head(10)
print(top_scaled_max[['Feature', 'Raw_IQR (Q75-Q25)', 'RobustScaler_scale_', 'Raw_Max', 'Scaled_Max']].to_string(index=False))


ROBUSTSCALER (NO CLIPPING) - SUMMARY STATISTICS
Train Scaled Min:     -6.9904e+06 | Max:      7.4200e+07 | Std:      1.0611e+06
Test  Scaled Min:     -1.1838e+04 | Max:      6.6500e+07 | Std:      1.0633e+06



10 FEATURES WITH THE SMALLEST IQR (AND THEIR SCALED IMPACT)
          Feature  Raw_IQR (Q75-Q25)  RobustScaler_scale_  Median    Raw_Max  Scaled_Max
       Active Std                0.0                  1.0     0.0 74200000.0  74200000.0
         Idle Std                0.0                  1.0     0.0 71300000.0  71300000.0
    Fwd PSH Flags                0.0                  1.0     0.0        1.0         1.0
   FIN Flag Count                0.0                  1.0     0.0        1.0         1.0
   URG Flag Count                0.0                  1.0     0.0        1.0         1.0
    Down/Up Ratio                1.0                  1.0     1.0       39.0        38.0
   PSH Flag Count                1.0                  1.0     0.0        1.0         1.0
   ACK Flag Count                1.0                  1.0     0.0        1.0         1.0
 act_data_pkt_fwd                2.0                  2.0     1.0     5666.0      2832.5
Total Fwd Packets                5.0             

## 3. StandardScaler with Clipping to [-10, 10] Comparison
For direct comparison, we fit `StandardScaler` strictly on `X_train` and apply clipping to `[-10, 10]`:
$$\hat{x}_{\text{clipped}} = \text{clip}\left(\frac{x - \mu}{\sigma}, -10, 10\right)$$
This bounds all inputs, eliminating extreme outlier gradients while preserving standard relative variance for non-outlier samples.


In [4]:
ss = StandardScaler()
X_train_ss = np.clip(ss.fit_transform(X_train), -10, 10)
X_val_ss = np.clip(ss.transform(X_val), -10, 10)
X_test_ss = np.clip(ss.transform(X_test), -10, 10)

print("="*85)
print("STANDARDSCALER (CLIPPED TO [-10, 10]) - SUMMARY STATISTICS")
print("="*85)
print(f"Train Scaled Min: {X_train_ss.min():10.4f} | Max: {X_train_ss.max():10.4f} | Std: {X_train_ss.std():10.4f}")
print(f"Test  Scaled Min: {X_test_ss.min():10.4f} | Max: {X_test_ss.max():10.4f} | Std: {X_test_ss.std():10.4f}")


STANDARDSCALER (CLIPPED TO [-10, 10]) - SUMMARY STATISTICS
Train Scaled Min:   -10.0000 | Max:    10.0000 | Std:     0.8537
Test  Scaled Min:   -10.0000 | Max:    10.0000 | Std:     0.8529


## 4. 1D-CNN Architecture & Training Protocol
We use the exact `Fast1DCNN` architecture, square-root smoothed class-weighted loss, optimizer, scheduler, and `copy.deepcopy` validation checkpoint restoration from Notebook 05:
- **Architecture**: 1D Conv (32 channels) $\to$ BatchNorm $\to$ ReLU $\to$ MaxPool(2) $\to$ 1D Conv (64 channels) $\to$ BatchNorm $\to$ ReLU $\to$ AdaptiveAvgPool(1) $\to$ Linear(64, 32) $\to$ ReLU $\to$ Dropout(0.2) $\to$ Linear(32, 7)
- **Loss**: `nn.CrossEntropyLoss(weight=smoothed_weights)`
- **Optimizer**: `AdamW(lr=0.002, weight_decay=1e-4)`
- **Scheduler**: `ReduceLROnPlateau(mode='max', factor=0.5, patience=2)`
- **Epochs**: 20
- **Batch Size**: 512 for training, 1024 for validation/test
- **Checkpointing**: Tracks Validation Macro F1 and restores `copy.deepcopy(model.state_dict())` of the best epoch.


In [5]:
class Fast1DCNN(nn.Module):
    def __init__(self, in_features=44, num_classes=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        return self.net(x.unsqueeze(1))

def train_cnn_with_val(model, train_loader, val_loader, criterion, epochs=20, lr=0.002):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    
    best_val_macro_f1 = -1.0
    best_epoch = 1
    best_weights = None
    history = {"train_loss": [], "val_acc": [], "val_macro_f1": []}
    t_start = time.time()
    
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            logits = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(by)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        history["train_loss"].append(epoch_loss)
        
        # Validation evaluation
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for bx, by in val_loader:
                bx = bx.to(device)
                preds = torch.argmax(model(bx), dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_targets.extend(by.numpy())
                
        val_acc = accuracy_score(val_targets, val_preds)
        val_mf1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)
        history["val_acc"].append(val_acc)
        history["val_macro_f1"].append(val_mf1)
        
        scheduler.step(val_mf1)
        
        if val_mf1 > best_val_macro_f1:
            best_val_macro_f1 = val_mf1
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())
            
        print(f"    Epoch {epoch:02d}/{epochs:02d} | Train Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro F1: {val_mf1:.4f}")
        
    train_time = time.time() - t_start
    if best_weights is not None:
        model.load_state_dict(best_weights)
    history["best_epoch"] = best_epoch
    history["best_val_macro_f1"] = best_val_macro_f1
    history["train_time"] = train_time
    return model, history

def predict_dl(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for bx, _ in loader:
            bx = bx.to(device)
            preds = torch.argmax(model(bx), dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

# Compute square-root smoothed inverse class weights (identical to notebook 05)
class_counts = np.bincount(y_train)
raw_weights = np.sqrt(len(y_train) / class_counts)
smoothed_weights = raw_weights / np.mean(raw_weights)
weights_tensor = torch.tensor(smoothed_weights, dtype=torch.float32).to(device)
cnn_criterion = nn.CrossEntropyLoss(weight=weights_tensor)

print("Smoothed Class Weights for CrossEntropyLoss:")
for idx, cls in enumerate(classes):
    print(f"  {cls:15s}: count={class_counts[idx]:6d} | weight={smoothed_weights[idx]:.4f}")


Smoothed Class Weights for CrossEntropyLoss:
  BENIGN         : count= 70000 | weight=0.2298
  Bot            : count=  1006 | weight=1.9173
  Brute Force    : count=  6405 | weight=0.7598
  DDoS           : count= 17500 | weight=0.4597
  DoS            : count= 21000 | weight=0.4196
  PortScan       : count=  1369 | weight=1.6436
  Web Attack     : count=  1500 | weight=1.5701


## 5. Experiment 1: 1D-CNN with RobustScaler (No Clipping)
We train the 1D-CNN using the unclipped `RobustScaler` features.


In [6]:
print("="*85)
print("EXPERIMENT 1: TRAINING 1D-CNN WITH ROBUSTSCALER (NO CLIPPING)")
print("="*85)

train_loader_rs = DataLoader(
    TensorDataset(torch.tensor(X_train_rs, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)),
    batch_size=512, shuffle=True
)
val_loader_rs = DataLoader(
    TensorDataset(torch.tensor(X_val_rs, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long)),
    batch_size=1024, shuffle=False
)
test_loader_rs = DataLoader(
    TensorDataset(torch.tensor(X_test_rs, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)),
    batch_size=1024, shuffle=False
)

torch.manual_seed(42)
np.random.seed(42)

model_rs = Fast1DCNN(in_features=X_train.shape[1], num_classes=num_classes).to(device)
model_rs, history_rs = train_cnn_with_val(
    model_rs, train_loader_rs, val_loader_rs, cnn_criterion, epochs=20, lr=0.002
)

# Test Evaluation
preds_rs = predict_dl(model_rs, test_loader_rs)
acc_rs = accuracy_score(y_test, preds_rs)
mf1_rs = f1_score(y_test, preds_rs, average='macro', zero_division=0)
wf1_rs = f1_score(y_test, preds_rs, average='weighted', zero_division=0)

print("\n" + "="*85)
print(f"ROBUSTSCALER TEST RESULTS (Restored Best Epoch: {history_rs['best_epoch']}/20)")
print(f"  Test Accuracy: {acc_rs:.4f} ({acc_rs*100:.2f}%)")
print(f"  Test Macro F1: {mf1_rs:.4f}")
print(f"  Test Weighted F1: {wf1_rs:.4f}")
print("="*85)
print("\nClassification Report (RobustScaler):")
print(classification_report(y_test, preds_rs, target_names=classes, zero_division=0))


EXPERIMENT 1: TRAINING 1D-CNN WITH ROBUSTSCALER (NO CLIPPING)


    Epoch 01/20 | Train Loss: 1.6769 | Val Acc: 0.6011 | Val Macro F1: 0.1340


    Epoch 02/20 | Train Loss: 1.6388 | Val Acc: 0.6015 | Val Macro F1: 0.1337


    Epoch 03/20 | Train Loss: 1.6287 | Val Acc: 0.6017 | Val Macro F1: 0.1343


    Epoch 04/20 | Train Loss: 1.6205 | Val Acc: 0.6020 | Val Macro F1: 0.1338


...

    Epoch 17/20 | Train Loss: 1.5579 | Val Acc: 0.6016 | Val Macro F1: 0.1950


    Epoch 18/20 | Train Loss: 1.5564 | Val Acc: 0.2573 | Val Macro F1: 0.0933


    Epoch 19/20 | Train Loss: 1.5532 | Val Acc: 0.2399 | Val Macro F1: 0.0827


    Epoch 20/20 | Train Loss: 1.5502 | Val Acc: 0.1864 | Val Macro F1: 0.0615



ROBUSTSCALER TEST RESULTS (Restored Best Epoch: 17/20)
  Test Accuracy: 0.6005 (60.05%)
  Test Macro F1: 0.1876
  Test Weighted F1: 0.4744

Classification Report (RobustScaler):
              precision    recall  f1-score   support

      BENIGN       0.60      0.98      0.75     15000
         Bot       0.39      0.33      0.36       216
 Brute Force       0.00      0.00      0.00      1372
        DDoS       0.72      0.10      0.17      3750
         DoS       0.57      0.02      0.04      4500
    PortScan       0.00      0.00      0.00       294
  Web Attack       0.00      0.00      0.00       321

    accuracy                           0.60     25453
   macro avg       0.33      0.20      0.19     25453
weighted avg       0.56      0.60      0.47     25453



## 6. Experiment 2: 1D-CNN with StandardScaler (Clipped [-10, 10])
We train the identical 1D-CNN architecture on features scaled with `StandardScaler` and clipped to `[-10, 10]`. All hyperparameters, seeds, epochs, and class weights are strictly identical.


In [7]:
print("="*85)
print("EXPERIMENT 2: TRAINING 1D-CNN WITH STANDARDSCALER (CLIPPED [-10, 10])")
print("="*85)

train_loader_ss = DataLoader(
    TensorDataset(torch.tensor(X_train_ss, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)),
    batch_size=512, shuffle=True
)
val_loader_ss = DataLoader(
    TensorDataset(torch.tensor(X_val_ss, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long)),
    batch_size=1024, shuffle=False
)
test_loader_ss = DataLoader(
    TensorDataset(torch.tensor(X_test_ss, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)),
    batch_size=1024, shuffle=False
)

torch.manual_seed(42)
np.random.seed(42)

model_ss = Fast1DCNN(in_features=X_train.shape[1], num_classes=num_classes).to(device)
model_ss, history_ss = train_cnn_with_val(
    model_ss, train_loader_ss, val_loader_ss, cnn_criterion, epochs=20, lr=0.002
)

# Test Evaluation
preds_ss = predict_dl(model_ss, test_loader_ss)
acc_ss = accuracy_score(y_test, preds_ss)
mf1_ss = f1_score(y_test, preds_ss, average='macro', zero_division=0)
wf1_ss = f1_score(y_test, preds_ss, average='weighted', zero_division=0)

print("\n" + "="*85)
print(f"STANDARDSCALER (CLIPPED) TEST RESULTS (Restored Best Epoch: {history_ss['best_epoch']}/20)")
print(f"  Test Accuracy: {acc_ss:.4f} ({acc_ss*100:.2f}%)")
print(f"  Test Macro F1: {mf1_ss:.4f}")
print(f"  Test Weighted F1: {wf1_ss:.4f}")
print("="*85)
print("\nClassification Report (StandardScaler Clipped [-10, 10]):")
print(classification_report(y_test, preds_ss, target_names=classes, zero_division=0))


EXPERIMENT 2: TRAINING 1D-CNN WITH STANDARDSCALER (CLIPPED [-10, 10])


    Epoch 01/20 | Train Loss: 0.9626 | Val Acc: 0.8665 | Val Macro F1: 0.4967


    Epoch 02/20 | Train Loss: 0.4440 | Val Acc: 0.9304 | Val Macro F1: 0.8042


    Epoch 03/20 | Train Loss: 0.3272 | Val Acc: 0.9286 | Val Macro F1: 0.8154


    Epoch 04/20 | Train Loss: 0.2799 | Val Acc: 0.9230 | Val Macro F1: 0.8082


...

    Epoch 17/20 | Train Loss: 0.1456 | Val Acc: 0.9652 | Val Macro F1: 0.8901


    Epoch 18/20 | Train Loss: 0.1445 | Val Acc: 0.9645 | Val Macro F1: 0.8949


    Epoch 19/20 | Train Loss: 0.1432 | Val Acc: 0.9626 | Val Macro F1: 0.8790


    Epoch 20/20 | Train Loss: 0.1391 | Val Acc: 0.9624 | Val Macro F1: 0.8861



STANDARDSCALER (CLIPPED) TEST RESULTS (Restored Best Epoch: 13/20)
  Test Accuracy: 0.9642 (96.42%)
  Test Macro F1: 0.9047
  Test Weighted F1: 0.9647

Classification Report (StandardScaler Clipped [-10, 10]):
              precision    recall  f1-score   support

      BENIGN       0.99      0.96      0.97     15000
         Bot       0.79      0.84      0.82       216
 Brute Force       0.92      0.96      0.94      1372
        DDoS       0.99      1.00      1.00      3750
         DoS       0.93      0.98      0.95      4500
    PortScan       0.75      0.89      0.81       294
  Web Attack       0.78      0.93      0.84       321

    accuracy                           0.96     25453
   macro avg       0.88      0.94      0.90     25453
weighted avg       0.97      0.96      0.96     25453



## 7. Comparative Analysis & Learning Curves
We plot the training loss and validation Macro F1 trajectories for both scaling approaches across the 20 epochs, alongside the final test metric comparison.


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, 21)

# Plot 1: Training Loss Trajectory
axes[0].plot(epochs_range, history_rs['train_loss'], 'r-o', label='RobustScaler (No Clip)')
axes[0].plot(epochs_range, history_ss['train_loss'], 'b-s', label='StandardScaler (Clip [-10, 10])')
axes[0].set_title('Training Loss per Epoch', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CrossEntropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Validation Macro F1 Trajectory
axes[1].plot(epochs_range, history_rs['val_macro_f1'], 'r-o', label='RobustScaler (No Clip)')
axes[1].plot(epochs_range, history_ss['val_macro_f1'], 'b-s', label='StandardScaler (Clip [-10, 10])')
axes[1].set_title('Validation Macro F1 per Epoch', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro F1 Score')
axes[1].set_ylim(0, 1.0)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Final Test Metric Comparison Bar Chart
comp_metrics = pd.DataFrame([
    {'Scaler': 'RobustScaler\n(No Clip)', 'Accuracy': acc_rs, 'Macro F1': mf1_rs, 'Weighted F1': wf1_rs},
    {'Scaler': 'StandardScaler\n(Clip [-10, 10])', 'Accuracy': acc_ss, 'Macro F1': mf1_ss, 'Weighted F1': wf1_ss}
])
melted = comp_metrics.melt(id_vars='Scaler', value_vars=['Accuracy', 'Macro F1', 'Weighted F1'], var_name='Metric', value_name='Score')
sns.barplot(data=melted, x='Scaler', y='Score', hue='Metric', ax=axes[2], palette=['#3498db', '#e74c3c', '#2ecc71'])
axes[2].set_title('Final Test Metric Comparison', fontsize=12, fontweight='bold')
axes[2].set_ylim(0, 1.1)
for p in axes[2].patches:
    h = p.get_height()
    if h > 0:
        axes[2].annotate(f'{h:.3f}', (p.get_x() + p.get_width() / 2., h + 0.02),
                         ha='center', va='bottom', fontsize=8, fontweight='bold')
axes[2].legend(loc='lower left')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../Models/scaling_reproduction_comparison.png', dpi=150)
plt.show()
print("Saved comparison visualization to ../Models/scaling_reproduction_comparison.png")


Saved comparison visualization to ../Models/scaling_reproduction_comparison.png


## 8. Export Summary Results to CSV
We save the comparison summary table to `../Models/scaling_reproduction.csv`.


In [9]:
os.makedirs("../Models", exist_ok=True)

# Build per-class F1 dictionary for both models
per_class_f1_rs = f1_score(y_test, preds_rs, average=None, zero_division=0)
per_class_f1_ss = f1_score(y_test, preds_ss, average=None, zero_division=0)

results = [
    {
        "Scaling Method": "RobustScaler (No Clipping)",
        "Train Min": float(X_train_rs.min()),
        "Train Max": float(X_train_rs.max()),
        "Train Std": float(X_train_rs.std()),
        "Test Min": float(X_test_rs.min()),
        "Test Max": float(X_test_rs.max()),
        "Test Std": float(X_test_rs.std()),
        "Test Accuracy": float(acc_rs),
        "Test Macro F1": float(mf1_rs),
        "Test Weighted F1": float(wf1_rs),
        "Best Checkpoint Epoch": int(history_rs['best_epoch']),
        "Best Val Macro F1": float(history_rs['best_val_macro_f1']),
        "Training Time (s)": float(f"{history_rs['train_time']:.2f}")
    },
    {
        "Scaling Method": "StandardScaler (Clipped [-10, 10])",
        "Train Min": float(X_train_ss.min()),
        "Train Max": float(X_train_ss.max()),
        "Train Std": float(X_train_ss.std()),
        "Test Min": float(X_test_ss.min()),
        "Test Max": float(X_test_ss.max()),
        "Test Std": float(X_test_ss.std()),
        "Test Accuracy": float(acc_ss),
        "Test Macro F1": float(mf1_ss),
        "Test Weighted F1": float(wf1_ss),
        "Best Checkpoint Epoch": int(history_ss['best_epoch']),
        "Best Val Macro F1": float(history_ss['best_val_macro_f1']),
        "Training Time (s)": float(f"{history_ss['train_time']:.2f}")
    }
]

# Add per-class F1 scores
for idx, cls in enumerate(classes):
    results[0][f"{cls} F1"] = float(per_class_f1_rs[idx])
    results[1][f"{cls} F1"] = float(per_class_f1_ss[idx])

summary_df = pd.DataFrame(results)
csv_out_path = "../Models/scaling_reproduction.csv"
summary_df.to_csv(csv_out_path, index=False)

print("="*110)
print("FINAL SCALING REPRODUCTION RESULTS TABLE")
print("="*110)
display(summary_df)
print(summary_df.to_string(index=False))
print(f"\nSaved results summary to: {csv_out_path}")


FINAL SCALING REPRODUCTION RESULTS TABLE


,Scaling Method,Train Min,Train Max,Train Std,Test Min,Test Max,Test Std,Test Accuracy,Test Macro F1,Test Weighted F1,Best Checkpoint Epoch,Best Val Macro F1,Training Time (s),BENIGN F1,Bot F1,Brute Force F1,DDoS F1,DoS F1,PortScan F1,Web Attack F1
0,RobustScaler (No Clipping),-6.990444e+06,74200000.0,1.061053e+06,-11838.007637,66500000.0,1.063272e+06,0.600519,0.187618,0.474390,17,0.195008,59.01,0.745310,0.358209,0.000000,0.168669,0.041140,0.000000,0.00000
1,"StandardScaler (Clipped [-10, 10])",-1.000000e+01,10.0,8.537074e-01,-10.000000,10.0,8.529186e-01,0.964248,0.904682,0.964739,13,0.913518,59.36,0.970713,0.817978,0.936094,0.995206,0.953594,0.814241,0.84495


                    Scaling Method     Train Min  Train Max    Train Std      Test Min   Test Max     Test Std  Test Accuracy  Test Macro F1  Test Weighted F1  Best Checkpoint Epoch  Best Val Macro F1  Training Time (s)  BENIGN F1   Bot F1  Brute Force F1  DDoS F1   DoS F1  PortScan F1  Web Attack F1
        RobustScaler (No Clipping) -6.990444e+06 74200000.0 1.061053e+06 -11838.007637 66500000.0 1.063272e+06       0.600519       0.187618          0.474390                     17           0.195008              59.01   0.745310 0.358209        0.000000 0.168669 0.041140     0.000000        0.00000
StandardScaler (Clipped [-10, 10]) -1.000000e+01       10.0 8.537074e-01    -10.000000       10.0 8.529186e-01       0.964248       0.904682          0.964739                     13           0.913518              59.36   0.970713 0.817978        0.936094 0.995206 0.953594     0.814241        0.84495

Saved results summary to: ../Models/scaling_reproduction.csv
